# Dự đoán tử vong do tai nạn giao thông tại Canada


---
## 1. Thu thap & Mo ta Du lieu

**Nguon:** National Collision Database (NCDB) Canada, 1999-2017  
**Quy mo:** ~2.57 trieu vu va cham, ~6.77 trieu quan sat (person-level)  
**Bien so:** 23 cot thuoc 3 nhom - va cham (C_*), phuong tien (V_*), ca nhan (P_*)


Thêm R và gói MICE vào để điền dữ liệu khuyết thiếu

In [ ]:
import rpy2;
%load_ext rpy2.ipython

In [ ]:
!Rscript -e "install.packages(c('mice','ROSE'), repos='https://cloud.r-project.org')"

In [ ]:
%%R
library(mice)

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 200
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["figure.figsize"] = (8, 5)
C0, C1, C2, C3, C4, C5 = sns.color_palette("muted", 6)
DPI = 300

DATA_DIR = "/kaggle/input"
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

print("Environment ready.")

In [ ]:
import glob

files = sorted(glob.glob(f"{DATA_DIR}/**/*.csv*", recursive=True))

df_list = []

for file in files:
    df = pd.read_csv(file, dtype=str, low_memory=False)
    df_list.append(df)
    print(f"{os.path.basename(file):20} {len(df):,} dòng")

df_full = pd.concat(df_list, ignore_index=True)

print(f"\nTổng: {len(df_full):,} dòng")

In [ ]:
NA_CODES = {"U", "UU", "UUUU", "X", "XX", "XXXX", "N", "NN", "NNNN", "Q", "QQ"}

def clean_numeric(series):
    return pd.to_numeric(series.replace(NA_CODES, np.nan), errors="coerce")


In [ ]:
n_collisions = df_full['C_CASE'].nunique()
print(f'So vu va cham: {n_collisions:,}')
print(f'So quan sat: {len(df_full):,}')


In [ ]:
df_full.head()

In [ ]:
NA_CODES = {'U', 'X', 'UU', 'XX', 'NN', 'QQ', 'NNNN'}

mask = df_full.isin(NA_CODES)

cols_with_na_codes = mask.any().loc[lambda x: x == True].index.tolist()

print("Cac cot co NA codes:")
print(cols_with_na_codes)

### Mo ta cac bien so

| Nhom | Bien | Mo ta |
|------|------|------|
| **Va cham** | C_YEAR | Nam xay ra |
| | C_MNTH | Thang |
| | C_WDAY | Ngay trong tuan |
| | C_HOUR | Gio |
| | C_SEV | Muc do nghiem trong (1=thuong vong, 2=thiet hai tai san) |
| | C_VEHS | So phuong tien |
| | C_CONF | Cau hinh va cham |
| | C_RCFG | Cau hinh duong |
| | C_WTHR | Thoi tiet |
| | C_RSUR | Be mat duong |
| | C_RALN | Can chinh duong |
| | C_TRAF | Kiem soat giao thong |
| **Phuong tien** | V_TYPE | Loai xe |
| | V_YEAR | Nam san xuat |
| **Ca nhan** | P_SEX | Gioi tinh |
| | P_AGE | Tuoi |
| | P_PSN | Vi tri ngoi |
| | P_ISEV | Muc do thuong tich |
| | P_SAFE | Thiet bi an toan |
| | P_USER | Loai nguoi tham gia giao thong |


---
## 2. Chuan bi du lieu


### 2.0. Lam sach du lieu
Du lieu NCDB su dung cac ma dac biet cho gia tri thieu: UU, NN, QQ, U, N.
Chuyen doi cac cot so sang numeric, xu ly ma thieu thanh NaN.


In [ ]:
num_cols = ['C_YEAR', 'C_MNTH', 'C_WDAY', 'C_HOUR', 'C_SEV', 'C_VEHS',
            'C_CONF', 'C_RCFG', 'C_WTHR', 'C_RSUR', 'C_RALN', 'C_TRAF',
            'V_ID', 'V_TYPE', 'V_YEAR', 'P_ID', 'P_AGE', 'P_PSN',
            'P_SEX', 'P_SAFE', 'P_ISEV', 'P_USER']

df_full['P_SEX'] = df_full['P_SEX'].replace({'M': 1, 'F': 0})

for col in num_cols:
    if col in df_full.columns:
        df_full[col] = clean_numeric(df_full[col])
        n_miss = df_full[col].isnull().sum()
        if n_miss > 0:
            print(f'  {col}: {n_miss} missing ({n_miss/len(df_full)*100:.2f}%)')
missing_before = {}
for col in num_cols:
    if col in df_full.columns:
        n = int(df_full[col].isnull().sum())
        if n > 0:
            missing_before[col] = n

# ---- Bar chart missing values ----
fig, ax = plt.subplots(figsize=(10, 6))
cols = list(missing_before.keys())
vals = list(missing_before.values())
colors = plt.cm.coolwarm(np.linspace(0.3, 0.9, len(cols)))
ax.barh(range(len(cols)), vals, color=colors)
ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols)
ax.set_xlabel('Missing count')
for i, v in enumerate(vals):
    ax.text(v + max(vals)*0.01, i, f'{v:,} ({v/len(df_full)*100:.2f}%)',
            va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'missing_before.png'), bbox_inches='tight')
plt.show()
print('Saved: missing_before.png')

### 2.0. Tao bien muc tieu (Fatality)
P_ISEV: 1=Khong thuong tich, 2=Thuong tich nhe, 3=Tu vong
Gop thanh phan loai nhi phan: **Fatality (1)** va **Non-Fatality (0)**


In [ ]:
print('Phan phoi P_ISEV (muc do thuong tich ca nhan):')
print(df_full['P_ISEV'].value_counts().sort_index())

df_full['Fatality'] = (df_full['P_ISEV'] == 3).astype(int)

print('\nPhan phoi bien muc tieu Fatality:')
print(df_full['Fatality'].value_counts())
print(f'\nTy le tu vong: {df_full["Fatality"].mean()*100:.3f}%')


---
## 2. Configuration & Experiment Setup

In [ ]:
# ============================================================
# CONFIGURATION - Thay doi thong so tai day
# ============================================================
SAMPLE_SIZE = 100000
SMOKE_SIZE = 2000
RANDOM_STATE_SAMPLE = 1
SEEDS = [1, 2, 3]

In [ ]:
from sklearn.model_selection import train_test_split

def take_sample(df, size, random_state):
    df = df.copy()
    df = df.dropna(subset=['Fatality'])
    valid_strata = df['Fatality'].value_counts()
    valid_strata = valid_strata[valid_strata >= 2].index
    df = df[df['Fatality'].isin(valid_strata)]
    actual_size = min(size, len(df))
    df_sample, _ = train_test_split(
        df, train_size=actual_size,
        stratify=df['Fatality'], random_state=random_state
    )
    return df_sample.reset_index(drop=True)

---
## 3. Imputation Pipeline Functions

In [ ]:
def story_imputation(df):
    rain_codes = [3, 4, 5]
    mask_rain = df['C_WTHR'].isin(rain_codes) & (df['C_RSUR'].isnull() | (df['C_RSUR'] == 9))
    df.loc[mask_rain, 'C_RSUR'] = 2
    mask_clear_sunny = (df['C_WTHR'] == 1) & (df['C_RSUR'].isnull() | (df['C_RSUR'] == 9))
    df.loc[mask_clear_sunny, 'C_RSUR'] = 1
    mask_dry = df['C_RSUR'] == 1
    df.loc[mask_dry & (df['C_WTHR'].isnull()), 'C_WTHR'] = 1
    max_vid_per_case = df.groupby('C_CASE')['V_ID'].transform('max')
    df['C_VEHS'] = df['C_VEHS'].fillna(max_vid_per_case)
    pos_to_user = {
        11: 1, 12: 2, 13: 2, 14: 2, 15: 2, 16: 2, 17: 2, 18: 2, 19: 2,
        21: 3, 22: 3, 23: 3, 24: 3, 25: 3, 26: 3, 27: 3, 28: 3, 29: 3,
        31: 4, 32: 4, 33: 4, 34: 4, 35: 4, 36: 4, 37: 4, 38: 4, 39: 4,
    }
    mask_pos = df['P_USER'].isnull() | (df['P_USER'].isin([9]))
    df.loc[mask_pos, 'P_USER'] = df.loc[mask_pos, 'P_PSN'].map(pos_to_user)
    print(f'Story-based: rain->wet={mask_rain.sum()}, dry->clear={mask_dry.sum()}, clear->dry={mask_clear_sunny.sum()}, P_USER_fill={mask_pos.sum()}')
    return df

In [ ]:
def filter_data(df):
    n_before = len(df)
    df = df.dropna(subset=['C_CASE', 'P_ISEV'])
    df = df[~((df['P_PSN'] == 99) & (df['V_YEAR'].isnull()))]
    df = df.reset_index(drop=True)
    print(f'Filter: {n_before:,} -> {len(df):,} (removed {(1-len(df)/n_before)*100:.2f}%)')
    return df

In [ ]:
def mice_imputation(df, num_cols, n_iter=5):
    import rpy2.robjects as ro
    import tempfile, os

    mice_all_cols = ['C_WTHR', 'C_RSUR', 'C_CONF', 'C_RCFG', 'C_RALN', 'C_TRAF', 'V_TYPE', 'P_SEX', 'P_USER', 'P_PSN', 'P_SAFE']

    tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
    tmp_path = tmp.name
    tmp.close()

    df[mice_all_cols].to_csv(tmp_path, index=False)

    ro.globalenv['csv_path'] = tmp_path
    ro.globalenv['m'] = n_iter

    ro.r('''
        library(mice)
        r_df <- read.csv(csv_path, na.strings = c("NA", ""))
        for (col in names(r_df)) {
            r_df[[col]] <- as.factor(r_df[[col]])
        }
        suppressMessages({
            imp <- mice(r_df, method = rep("polyreg", ncol(r_df)),
                        m = 1, maxit = m, seed = 42, printFlag = FALSE)
        })
        filled <- complete(imp)
        write.csv(filled, csv_path, row.names = FALSE)
    ''')

    filled_py = pd.read_csv(tmp_path)
    os.unlink(tmp_path)

    for col in mice_all_cols:
        df[col] = filled_py[col].values.astype(int)

    missing_counts = {}
    for col in num_cols:
        if col in df.columns:
            n = int(df[col].isnull().sum())
            if n > 0:
                missing_counts[col] = n
    print(f'MICE (R polyreg) imputed: {mice_all_cols}')
    print(f'Remaining missing: {missing_counts}')
    return df

---
## 4. Model Pipeline Functions

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from scipy.sparse import hstack, csr_matrix

def prepare_features(df, feature_cols, nominal_cols, categorical_cols, target):
    df_model = df[feature_cols + nominal_cols + [target]].copy()
    for col in categorical_cols:
        df_model[col] = df_model[col].astype(str)
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col])
    df_model = df_model.dropna().reset_index(drop=True)
    X_dense = csr_matrix(df_model[feature_cols].values)
    ohe = OneHotEncoder(sparse_output=True, min_frequency=0.001, handle_unknown='infrequent_if_exist')
    X_ohe_sparse = ohe.fit_transform(df_model[nominal_cols])
    X_lasso = hstack([X_dense, X_ohe_sparse], format='csr')
    lasso_feature_names = feature_cols + list(ohe.get_feature_names_out())
    X_xgb = df_model[feature_cols + nominal_cols].values
    y = df_model[target].astype(int).values
    print(f'Lasso: {len(lasso_feature_names)} feats | XGBoost: {X_xgb.shape[1]} feats | Samples: {len(y):,}')
    return X_lasso, X_xgb, y, lasso_feature_names

In [ ]:
from sklearn.neighbors import NearestNeighbors

def rose_r(X_train, y_train, sampling_strategy, seed):
    import rpy2.robjects as ro
    import tempfile, os
    import pandas as pd

    n_maj = int(np.sum(y_train == 0))
    n_min = int(np.sum(y_train == 1))

    if sampling_strategy < 1:
        n_synth = max(0, int(n_maj * sampling_strategy) - n_min)
    else:
        n_synth = max(0, int(n_min * sampling_strategy) - n_min)

    if n_synth <= 0:
        return X_train, y_train

    n_total = n_maj + n_min + n_synth

    tmp = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
    tmp_path = tmp.name
    tmp.close()

    if hasattr(X_train, 'toarray'):
        X_dense = X_train.toarray()
    else:
        X_dense = X_train
    df = pd.DataFrame(X_dense)
    df['y'] = y_train.astype(int)
    df.to_csv(tmp_path, index=False)

    ro.globalenv['csv_path'] = tmp_path
    ro.globalenv['n_total'] = int(n_total)
    ro.globalenv['seed_val'] = int(seed)

    ro.r('''
        library(ROSE)
        data <- read.csv(csv_path)
        data$y <- as.factor(data$y)
        result <- ROSE(y ~ ., data = data, N = n_total, seed = seed_val)
        rose_data <- result$data
        write.csv(rose_data, csv_path, row.names = FALSE)
    ''')

    rose_df = pd.read_csv(tmp_path)
    os.unlink(tmp_path)

    X_rose = rose_df.drop(columns=['y']).values.astype(float)
    y_rose = rose_df['y'].values.astype(int)
    return X_rose, y_rose


def apply_sampling(X_train, y_train, random_state=42):
    from imblearn.under_sampling import RandomUnderSampler
    rus = RandomUnderSampler(random_state=random_state)
    X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

    X_train_rose, y_train_rose = rose_r(X_train, y_train, 0.5, random_state)

    rus2 = RandomUnderSampler(random_state=random_state, sampling_strategy=0.4)
    rose2 = lambda X, y: rose_r(X, y, 0.8, random_state)
    X_train_mix, y_train_mix = rus2.fit_resample(X_train, y_train)
    X_train_mix, y_train_mix = rose2(X_train_mix, y_train_mix)

    print(f'  No sampling: {X_train.shape[0]:,} (fatal={y_train.sum():,})')
    print(f'  Undersampling: {X_train_under.shape[0]:,} (fatal={y_train_under.sum():,})')
    print(f'  ROSE: {X_train_rose.shape[0]:,} (fatal={y_train_rose.sum():,})')
    print(f'  Mixed: {X_train_mix.shape[0]:,} (fatal={y_train_mix.sum():,})')
    return X_train_under, y_train_under, X_train_rose, y_train_rose, X_train_mix, y_train_mix

In [ ]:
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, average_precision_score, matthews_corrcoef

def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    prauc = average_precision_score(y_test, y_prob)
    g_mean = np.sqrt(sens * spec) if (sens * spec) >= 0 else 0
    print(f'    {name:30s} | Acc={acc:.3f} | Sens={sens:.3f} | Prec={prec:.3f} | F1={f1:.3f} | MCC={mcc:.3f} | G={g_mean:.3f} | AUC={auc:.3f} | PR={prauc:.3f}')
    return {'Accuracy': acc, 'Sensitivity': sens, 'Specificity': spec,
            'Precision': prec, 'F1': f1, 'MCC': mcc, 'G_mean': g_mean, 'AUC_ROC': auc, 'PR_AUC': prauc}

def run_experiment(X_lasso_train, X_lasso_test, X_raw_train, X_raw_test,
                   y_train, y_test, random_state=42):
    results = []
    best_xgb = None
    # === Lasso (OHE features) ===
    X_tr_u, y_tr_u, X_tr_r, y_tr_r, X_tr_m, y_tr_m = apply_sampling(X_lasso_train, y_train, random_state)
    for sname, X_tr, y_tr in [
        ('No Sampling', X_lasso_train, y_train),
        ('Undersampling', X_tr_u, y_tr_u),
        ('ROSE', X_tr_r, y_tr_r),
        ('Mixed Sampling', X_tr_m, y_tr_m),
    ]:
        lasso = LogisticRegressionCV(penalty='l1', solver='saga', Cs=10, cv=5, max_iter=1000, random_state=random_state, n_jobs=-1)
        lasso.fit(X_tr, y_tr)
        res = evaluate_model(f'Lasso + {sname}', lasso, X_lasso_test, y_test)
        res['Model'] = f'Lasso + {sname}'; results.append(res)
    # === Tree-based (raw 18 features) ===
    X_tr_u, y_tr_u, X_tr_r, y_tr_r, X_tr_m, y_tr_m = apply_sampling(X_raw_train, y_train, random_state)
    sampling_data = [
        ('No Sampling', X_raw_train, y_train),
        ('Undersampling', X_tr_u, y_tr_u),
        ('ROSE', X_tr_r, y_tr_r),
        ('Mixed Sampling', X_tr_m, y_tr_m),
    ]
    # XGBoost
    from xgboost import XGBClassifier
    for sname, X_tr, y_tr in sampling_data:
        spw = 1 if sname == 'No Sampling' else max(1, sum(y_tr == 0) / max(sum(y_tr == 1), 1))
        xgb = XGBClassifier(n_estimators=1200, learning_rate=0.5, max_depth=3, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw, random_state=random_state, use_label_encoder=False, eval_metric='logloss', n_jobs=-1)
        xgb.fit(X_tr, y_tr)
        res = evaluate_model(f'XGBoost + {sname}', xgb, X_raw_test, y_test)
        res['Model'] = f'XGBoost + {sname}'; results.append(res)
        if sname == 'Mixed Sampling': best_xgb = xgb
    # LightGBM
    import lightgbm as lgb
    for sname, X_tr, y_tr in sampling_data:
        lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.1, max_depth=5, class_weight='balanced', random_state=random_state, n_jobs=-1, verbose=-1)
        lgb_model.fit(X_tr, y_tr)
        res = evaluate_model(f'LightGBM + {sname}', lgb_model, X_raw_test, y_test)
        res['Model'] = f'LightGBM + {sname}'; results.append(res)
    # Random Forest
    for sname, X_tr, y_tr in sampling_data:
        rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced', random_state=random_state, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        res = evaluate_model(f'RandomForest + {sname}', rf, X_raw_test, y_test)
        res['Model'] = f'RandomForest + {sname}'; results.append(res)
    return results, best_xgb

In [ ]:
feature_cols = [
    'C_YEAR', 'C_MNTH', 'C_WDAY', 'C_HOUR', 'C_VEHS',
    'V_TYPE', 'V_YEAR', 'P_SEX', 'P_AGE', 'P_PSN', 'P_USER'
]
nominal_cols = ['C_CONF', 'C_RCFG', 'C_RALN', 'C_TRAF', 'C_WTHR', 'C_RSUR', 'P_SAFE']
categorical_cols = ['V_TYPE', 'P_SEX']
target = 'Fatality'

def simple_imputation(df):
    for col in ['P_AGE', 'V_YEAR', 'C_HOUR']:
        df[col] = df[col].fillna(df[col].median())
    for col in ['C_MNTH', 'C_WDAY']:
        df[col] = df[col].fillna(df[col].mode()[0])
    all_feat = feature_cols + nominal_cols
    for col in all_feat:
        if col in df.columns and df[col].isna().sum() > 0:
            mode_val = df[col].mode(dropna=True)
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val[0])
    return df

In [ ]:
from imblearn.over_sampling import SMOTE, SMOTENC, BorderlineSMOTE
from imblearn.combine import SMOTETomek

CAT_INDICES_RAW = [1, 2, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17]

def apply_sampling_v2(X_train, y_train, sampling_name, random_state=42, feat_type='raw'):
    from imblearn.under_sampling import RandomUnderSampler
    if sampling_name == 'no':
        return X_train, y_train
    elif sampling_name == 'under':
        return RandomUnderSampler(random_state=random_state).fit_resample(X_train, y_train)
    elif sampling_name == 'rose':
        return rose_r(X_train, y_train, 0.5, random_state)
    elif sampling_name == 'smote':
        return SMOTE(random_state=random_state).fit_resample(X_train, y_train)
    elif sampling_name == 'smotenc':
        if feat_type == 'ohe':
            return SMOTE(random_state=random_state).fit_resample(X_train, y_train)
        return SMOTENC(categorical_features=CAT_INDICES_RAW, random_state=random_state).fit_resample(X_train, y_train)
    elif sampling_name == 'borderline':
        return BorderlineSMOTE(random_state=random_state).fit_resample(X_train, y_train)
    elif sampling_name == 'smote_tomek':
        return SMOTETomek(random_state=random_state).fit_resample(X_train, y_train)
    elif sampling_name == 'adasyn':
        from imblearn.over_sampling import ADASYN
        return ADASYN(random_state=random_state).fit_resample(X_train, y_train)
    elif sampling_name == 'kmeans_smote':
        from imblearn.over_sampling import KMeansSMOTE
        return KMeansSMOTE(random_state=random_state, kmeans_ratio=0.1, cluster_balance_threshold=0.1).fit_resample(X_train, y_train)
    elif sampling_name == 'nearmiss':
        from imblearn.under_sampling import NearMiss
        return NearMiss(version=1).fit_resample(X_train, y_train)
    elif sampling_name == 'mixed':
        rus = RandomUnderSampler(random_state=random_state, sampling_strategy=0.4)
        X_mix, y_mix = rus.fit_resample(X_train, y_train)
        return rose_r(X_mix, y_mix, 0.8, random_state)
    else:
        return X_train, y_train


In [ ]:
print('='*60)
print('  SMOKE TEST - Verify pipeline with small sample')
print('='*60)

df_smoke = take_sample(df_full, SMOKE_SIZE, 1)
df_smoke = story_imputation(df_smoke)
df_smoke = filter_data(df_smoke)
df_smoke = mice_imputation(df_smoke, num_cols)
df_smoke = simple_imputation(df_smoke)

X_lasso, X_xgb, y, _ = prepare_features(df_smoke, feature_cols, nominal_cols, categorical_cols, target)
indices = np.arange(len(y))
train_idx, test_idx, y_train, y_test = train_test_split(indices, y, test_size=0.3, random_state=1, stratify=y)
X_lo_tr, X_lo_te = X_lasso[train_idx], X_lasso[test_idx]
X_ra_tr, X_ra_te = X_xgb[train_idx], X_xgb[test_idx]

_, _ = run_experiment(X_lo_tr, X_lo_te, X_ra_tr, X_ra_te, y_train, y_test, 1)
print('\n✅ Smoke test passed - pipeline ready')

---
## PART 1: TAI HIEN (4 sampling x 2 models: Lasso + XGBoost)

In [ ]:
all_results_p1 = []

for i, seed in enumerate(SEEDS):
    print(f'\n{"="*60}')
    print(f'  PART 1 | Seed {seed} ({i+1}/{len(SEEDS)})')
    print(f'{"="*60}')

    df = take_sample(df_full, SAMPLE_SIZE, seed)
    df = story_imputation(df)
    df = filter_data(df)
    df = mice_imputation(df, num_cols)
    df = simple_imputation(df)

    X_lasso, X_xgb, y, lasso_feature_names = prepare_features(
        df, feature_cols, nominal_cols, categorical_cols, target)

    indices = np.arange(len(y))
    train_idx, test_idx, y_train, y_test = train_test_split(
        indices, y, test_size=0.3, random_state=seed, stratify=y)
    X_lo_tr, X_lo_te = X_lasso[train_idx], X_lasso[test_idx]
    X_ra_tr, X_ra_te = X_xgb[train_idx],   X_xgb[test_idx]

    results_p1, _ = run_experiment(X_lo_tr, X_lo_te, X_ra_tr, X_ra_te, y_train, y_test, seed)
    for r in results_p1:
        r['Seed'] = seed
    all_results_p1.extend(results_p1)
    print(f'  Part 1 seed {seed} done')

### PART 1: Ket qua trung binh qua cac seeds

In [ ]:
df_p1 = pd.DataFrame(all_results_p1)
metric_cols = ['Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1', 'MCC', 'G_mean', 'AUC_ROC', 'PR_AUC']
agg_p1 = df_p1.groupby('Model')[metric_cols].agg(['mean', 'std']).round(4)
agg_p1.columns = ['_'.join(c) for c in agg_p1.columns]
agg_p1 = agg_p1.reset_index()
print('=' * 70)
print('PART 1 - 4 sampling x 2 models (Lasso + XGBoost)')
print('=' * 70)
print(agg_p1.to_string(index=False))
df_p1.to_csv(os.path.join(OUT_DIR, 'part1_results.csv'), index=False)
agg_p1.to_csv(os.path.join(OUT_DIR, 'part1_summary.csv'), index=False)

---
## PART 2: CAI TIEN (SMOTE variants x 5 models: Lasso + XGBoost + LightGBM + RF + CatBoost)

In [ ]:
from imblearn.base import BaseSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline as SkPipeline

class ROSESampler(BaseSampler):
    def __init__(self, sampling_strategy=0.5, random_state=None):
        super().__init__(sampling_strategy=sampling_strategy)
        self.random_state = random_state

    def _fit_resample(self, X, y):
        return rose_r(X, y, self.sampling_strategy, self.random_state)

class MixedSampler(BaseSampler):
    def __init__(self, sampling_strategy='auto', random_state=None):
        super().__init__(sampling_strategy=sampling_strategy)
        self.random_state = random_state

    def _fit_resample(self, X, y):
        from imblearn.under_sampling import RandomUnderSampler
        rus = RandomUnderSampler(random_state=self.random_state, sampling_strategy=0.4)
        X_mix, y_mix = rus.fit_resample(X, y)
        return rose_r(X_mix, y_mix, 0.8, self.random_state)

def get_sampler(sname, seed):
    from imblearn.over_sampling import SMOTE, SMOTENC, BorderlineSMOTE, ADASYN, KMeansSMOTE
    from imblearn.combine import SMOTETomek
    from imblearn.under_sampling import RandomUnderSampler, NearMiss
    mapping = {
        'under': RandomUnderSampler(random_state=seed),
        'rose': ROSESampler(random_state=seed),
        'mixed': MixedSampler(random_state=seed),
        'smote': SMOTE(random_state=seed),
        'smotenc': SMOTE(random_state=seed),
        'borderline': BorderlineSMOTE(random_state=seed),
        'smote_tomek': SMOTETomek(random_state=seed),
        'adasyn': ADASYN(random_state=seed),
        'kmeans_smote': KMeansSMOTE(random_state=seed, kmeans_ratio=0.1, cluster_balance_threshold=0.1),
        'nearmiss': NearMiss(version=1),
    }
    return mapping.get(sname)

def get_classifier(model_name, seed, sampler_active):
    from sklearn.linear_model import LogisticRegression
    if model_name == 'Lasso':
        return LogisticRegression(penalty='l1', solver='saga', random_state=seed, n_jobs=-1)
    from xgboost import XGBClassifier
    if model_name == 'XGBoost':
        spw = 1 if sampler_active else 200
        return XGBClassifier(scale_pos_weight=spw, use_label_encoder=False,
                            eval_metric='logloss', random_state=seed, n_jobs=-1)
    import lightgbm as lgb
    if model_name == 'LightGBM':
        return lgb.LGBMClassifier(class_weight='balanced' if not sampler_active else None,
                                 random_state=seed, n_jobs=-1, verbose=-1)
    from sklearn.ensemble import RandomForestClassifier
    if model_name == 'RandomForest':
        return RandomForestClassifier(class_weight='balanced' if not sampler_active else None,
                                     random_state=seed, n_jobs=-1)
    from catboost import CatBoostClassifier
    if model_name == 'CatBoost':
        return CatBoostClassifier(random_seed=seed, verbose=0)
    return None

def build_pipe(sname, model_name, seed):
    sampler_active = sname != 'no'
    clf = get_classifier(model_name, seed, sampler_active)
    if sampler_active:
        sampler = get_sampler(sname, seed)
        return ImbPipeline([('sampler', sampler), ('clf', clf)])
    else:
        return SkPipeline([('clf', clf)])

def pipe_params(base_grid):
    return {f'clf__{k}': v for k, v in base_grid.items()}

def extract_clf_params(best_params):
    return {k.replace('clf__', ''): v for k, v in best_params.items() if k.startswith('clf__')}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression

SAMPLING_V2 = ['no', 'under', 'rose', 'mixed', 'smote', 'smotenc', 'borderline', 'smote_tomek', 'adasyn', 'kmeans_smote', 'nearmiss']

param_grids = {
    'Lasso': {
        'C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
        'max_iter': [1000, 5000],
        'solver': ['saga'],
        'penalty': ['l1'],
    },
    'XGBoost': {
        'learning_rate': [0.05, 0.1, 0.3, 0.5],
        'max_depth': [3, 5, 7],
        'n_estimators': [600, 900, 1200],
        'subsample': [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0],
    },
    'LightGBM': {
        'learning_rate': [0.05, 0.1, 0.2],
        'max_depth': [3, 5, -1],
        'num_leaves': [31, 63, 127],
        'n_estimators': [500, 1000],
    },
    'RandomForest': {
        'n_estimators': [200, 300, 500],
        'max_depth': [5, 10, 15, None],
        'min_samples_leaf': [1, 5, 10],
    },
    'CatBoost': {
        'learning_rate': [0.03, 0.05, 0.1],
        'depth': [4, 6, 8],
        'iterations': [500, 1000],
    },
}

all_results_p2 = []
best_params_store = {}

# Phase 1: Grid search on seed 1 (no data leakage: sampler inside CV pipeline)
seed = SEEDS[0]
print()
print("=" * 60)
print(f'  PART 2 | PHASE 1: Grid Search on seed {seed}')
print("=" * 60)

df = take_sample(df_full, SAMPLE_SIZE, seed)
df = story_imputation(df)
df = filter_data(df)
df = mice_imputation(df, num_cols)
df = simple_imputation(df)

X_lasso, X_xgb, y, lasso_feature_names = prepare_features(
    df, feature_cols, nominal_cols, categorical_cols, target)

indices = np.arange(len(y))
train_idx, test_idx, y_train, y_test = train_test_split(
    indices, y, test_size=0.3, random_state=seed, stratify=y)
X_lo_tr, X_lo_te = X_lasso[train_idx], X_lasso[test_idx]
X_ra_tr, X_ra_te = X_xgb[train_idx],   X_xgb[test_idx]

for sname in SAMPLING_V2:
    print(f'  --- Sampling: {sname} (grid search) ---')

    for mname in ['Lasso', 'XGBoost', 'LightGBM', 'RandomForest', 'CatBoost']:
        X_tr = X_lo_tr if mname == 'Lasso' else X_ra_tr
        X_te = X_lo_te if mname == 'Lasso' else X_ra_te

        pipe = build_pipe(sname, mname, seed)
        gs = RandomizedSearchCV(pipe, pipe_params(param_grids[mname]),
                                n_iter=10, cv=3, scoring='f1',
                                random_state=seed, n_jobs=-1)
        gs.fit(X_tr, y_train)
        best_clf_params = extract_clf_params(gs.best_params_)
        best_params_store[(sname, mname)] = best_clf_params

        res = evaluate_model(f'{mname} + {sname}', gs.best_estimator_, X_te, y_test)
        res['Model'] = f'{mname} + {sname}'; res['Seed'] = seed; all_results_p2.append(res)
        print(f'    {mname:12s} best: {best_clf_params}')

print(f'\nPhase 1 done. Best params stored for {len(best_params_store)} combos.')

# Phase 2: Evaluate remaining seeds with best params
for i, seed in enumerate(SEEDS[1:], 2):
    print()
    print("=" * 60)
    print(f'  PART 2 | PHASE 2: Seed {seed} ({i}/{len(SEEDS)})')
    print("=" * 60)

    df = take_sample(df_full, SAMPLE_SIZE, seed)
    df = story_imputation(df)
    df = filter_data(df)
    df = mice_imputation(df, num_cols)
    df = simple_imputation(df)

    X_lasso, X_xgb, y, lasso_feature_names = prepare_features(
        df, feature_cols, nominal_cols, categorical_cols, target)

    indices = np.arange(len(y))
    train_idx, test_idx, y_train, y_test = train_test_split(
        indices, y, test_size=0.3, random_state=seed, stratify=y)
    X_lo_tr, X_lo_te = X_lasso[train_idx], X_lasso[test_idx]
    X_ra_tr, X_ra_te = X_xgb[train_idx],   X_xgb[test_idx]

    for sname in SAMPLING_V2:
        print(f'  --- Sampling: {sname} ---')

        for mname in ['Lasso', 'XGBoost', 'LightGBM', 'RandomForest', 'CatBoost']:
            X_tr = X_lo_tr if mname == 'Lasso' else X_ra_tr
            X_te = X_lo_te if mname == 'Lasso' else X_ra_te

            pipe = build_pipe(sname, mname, seed)
            params = best_params_store[(sname, mname)]
            pipe.set_params(**{f'clf__{k}': v for k, v in params.items()})
            pipe.fit(X_tr, y_train)

            res = evaluate_model(f'{mname} + {sname}', pipe, X_te, y_test)
            res['Model'] = f'{mname} + {sname}'; res['Seed'] = seed; all_results_p2.append(res)

    print(f'  Part 2 seed {seed} done')

### PART 2: Ket qua trung binh qua cac seeds

In [ ]:
df_p2 = pd.DataFrame(all_results_p2)
agg_p2 = df_p2.groupby('Model')[metric_cols].agg(['mean', 'std']).round(4)
agg_p2.columns = ['_'.join(c) for c in agg_p2.columns]
agg_p2 = agg_p2.reset_index()
print('=' * 70)
print('PART 2 - SMOTE variants x 5 models (RandomizedSearchCV tuned)')
print('=' * 70)
print(agg_p2.to_string(index=False))
print(f'\nBest params tuned on seed {SEEDS[0]} via RandomizedSearchCV(cv=3, n_iter=10)')
df_p2.to_csv(os.path.join(OUT_DIR, 'part2_results.csv'), index=False)
agg_p2.to_csv(os.path.join(OUT_DIR, 'part2_summary.csv'), index=False)

---
## SO SANH: Part 1 vs Part 2

In [ ]:
# Top models + Best config selection
df_p1['Part'] = 'Part1'
df_p2['Part'] = 'Part2'
all_combined = pd.concat([df_p1, df_p2], ignore_index=True)

top_f1 = all_combined.groupby('Model')['F1'].mean().sort_values(ascending=False).head(10)
top_prauc = all_combined.groupby('Model')['PR_AUC'].mean().sort_values(ascending=False).head(10)

print('Top 10 models theo F1:')
for i, (m, v) in enumerate(top_f1.items(), 1):
    print(f'  {i:2d}. {m:30s}  F1={v:.4f}')

print('\nTop 10 models theo PR-AUC:')
for i, (m, v) in enumerate(top_prauc.items(), 1):
    print(f'  {i:2d}. {m:30s}  PR={v:.4f}')

best_model_name = top_f1.index[0]
print(f'\n{"="*60}')
print(f'BEST CONFIG: {best_model_name}')
print(f'  F1={top_f1.iloc[0]:.4f} | PR-AUC={top_prauc[best_model_name]:.4f}')
print(f'{"="*60}')
best_split = best_model_name.split(' + ', 1)
if len(best_split) == 2:
    best_model_type, best_sampling = best_split
else:
    best_model_type, best_sampling = 'XGBoost', 'Mixed Sampling'
print(f'  Model: {best_model_type} | Sampling: {best_sampling}')

best_config = {'Model': best_model_type, 'Sampling': best_sampling, 'FullName': best_model_name}
all_combined.to_csv(os.path.join(OUT_DIR, 'all_results_combined.csv'), index=False)

---
## FEATURE IMPORTANCE + KET LUAN

In [ ]:
# Feature importance from best config
if len(all_combined) > 0:
    from xgboost import XGBClassifier
    import lightgbm as lgb
    from sklearn.ensemble import RandomForestClassifier
    from catboost import CatBoostClassifier
    from sklearn.linear_model import LogisticRegression

    best_model_name = all_combined.groupby('Model')['F1'].mean().idxmax()
    best_parts = best_model_name.split(' + ', 1)
    if len(best_parts) == 2:
        best_mtype, best_samp = best_parts
    else:
        best_mtype, best_samp = 'XGBoost', 'Mixed Sampling'

    samp_key = best_samp.lower().replace(' ', '_')
    if samp_key == 'no_sampling': samp_key = 'no'
    if samp_key == 'undersampling': samp_key = 'under'
    if samp_key == 'mixed_sampling': samp_key = 'mixed'

    print(f'Feature importance using: {best_model_name}')

    df_imp = take_sample(df_full, min(50000, SAMPLE_SIZE), SEEDS[-1])
    df_imp = story_imputation(df_imp)
    df_imp = filter_data(df_imp)
    df_imp = mice_imputation(df_imp, num_cols)
    df_imp = simple_imputation(df_imp)
    X_lo, X_ra, y_imp, _ = prepare_features(df_imp, feature_cols, nominal_cols, categorical_cols, target)

    if best_mtype == 'Lasso':
        X_ra_s, y_ra_s = apply_sampling_v2(X_lo, y_imp, samp_key, 42, 'ohe')
        params = best_params_store.get((samp_key, 'Lasso'), {'C': 0.1})
        model = LogisticRegression(penalty='l1', solver='saga', random_state=42, n_jobs=-1, **params)
        model.fit(X_ra_s, y_ra_s)
        importances = np.abs(model.coef_[0])
        fi_names = lasso_feature_names
    else:
        X_ra_s, y_ra_s = apply_sampling_v2(X_ra, y_imp, samp_key, 42, 'raw')
        spw = max(1, sum(y_ra_s == 0) / max(sum(y_ra_s == 1), 1))
        if best_mtype == 'XGBoost':
            params = best_params_store.get((samp_key, 'XGBoost'), {'n_estimators': 1200})
            model = XGBClassifier(scale_pos_weight=spw, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1, **params)
        elif best_mtype == 'LightGBM':
            params = best_params_store.get((samp_key, 'LightGBM'), {'n_estimators': 1000})
            model = lgb.LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1, **params)
        elif best_mtype == 'RandomForest':
            params = best_params_store.get((samp_key, 'RandomForest'), {'n_estimators': 300})
            model = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1, **params)
        elif best_mtype == 'CatBoost':
            params = best_params_store.get((samp_key, 'CatBoost'), {'iterations': 1000})
            model = CatBoostClassifier(random_seed=42, verbose=0, **params)
        else:
            model = XGBClassifier(scale_pos_weight=spw, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
        model.fit(X_ra_s, y_ra_s)
        fi_names = feature_cols + nominal_cols
        importances = model.feature_importances_

    fi_df = pd.DataFrame({'Feature': fi_names, 'Importance': importances}).sort_values('Importance', ascending=False)
    print(f'\nTop 10 features ({best_model_name}):')
    for i, (_, row) in enumerate(fi_df.head(10).iterrows(), 1):
        print(f'  {i:2d}. {row["Feature"]:10s}  {row["Importance"]:.4f}')
    fig, ax = plt.subplots(figsize=(10, 6))
    top10 = fi_df.head(10)
    colors = sns.color_palette('viridis', n_colors=len(top10))
    ax.barh(range(len(top10)), top10['Importance'][::-1], color=colors[::-1])
    ax.set_yticks(range(len(top10))); ax.set_yticklabels(top10['Feature'][::-1])
    ax.set_xlabel('Feature Importance')
    for i, v in enumerate(top10['Importance'][::-1]):
        ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, '06_feature_importance.png'), bbox_inches='tight'); plt.show()
    print('\nSaved: 06_feature_importance.png')
    fi_df.to_csv(os.path.join(OUT_DIR, 'feature_importance.csv'), index=False)

# Final summary
best_model_name = all_combined.groupby('Model')['F1'].mean().idxmax() if len(all_combined) > 0 else 'N/A'
print()
print('=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)
config_summary = f"""
CONFIG:
  Sample size: {SAMPLE_SIZE:,}
  Seeds: {SEEDS}
  Grid Search: RandomizedSearchCV(cv=3, n_iter=10) on seed {SEEDS[0]}
  Part 1: {len(all_results_p1)} evaluations
  Part 2: {len(all_results_p2)} evaluations
  Best config: {best_model_name}
"""
print(config_summary)

---
## FULL DATASET RUN (6.77M records)
Chay best config tren toan bo du lieu, 1 lan 70/30 split, seed=42


In [ ]:
# === Full dataset run: best config ===
if len(all_combined) > 0:
    mtype = best_config['Model']
    samp = best_config['Sampling']
    samp_key = samp.lower().replace(' ', '_')
    if samp_key == 'no_sampling': samp_key = 'no'
    if samp_key == 'undersampling': samp_key = 'under'
    if samp_key == 'mixed_sampling': samp_key = 'mixed'

    print('Loading full dataset (6.77M)...')
    df = df_full.copy()
    df = story_imputation(df)
    df = filter_data(df)
    df = mice_imputation(df, num_cols)
    df = simple_imputation(df)

    X_lasso, X_xgb, y, lasso_feature_names = prepare_features(
        df, feature_cols, nominal_cols, categorical_cols, target)

    indices = np.arange(len(y))
    train_idx, test_idx, y_train, y_test = train_test_split(
        indices, y, test_size=0.3, random_state=42, stratify=y)
    print(f'Full: {len(y):,} | Train: {len(y_train):,} | Test: {len(y_test):,}')
    print(f'Fatality rate: {y.mean()*100:.3f}%')
    print(f'Best config: {best_config["FullName"]}')

    is_lasso = (mtype == 'Lasso')
    X_train = X_lasso[train_idx] if is_lasso else X_xgb[train_idx]
    X_test  = X_lasso[test_idx]  if is_lasso else X_xgb[test_idx]
    feat_t  = 'ohe' if is_lasso else 'raw'
    X_tr_s, y_tr_s = apply_sampling_v2(X_train, y_train, samp_key, 42, feat_t)
    spw = max(1, sum(y_tr_s == 0) / max(sum(y_tr_s == 1), 1))

    if mtype == 'Lasso':
        params = best_params_store.get((samp_key, 'Lasso'), {'C': 0.1})
        model = LogisticRegression(penalty='l1', solver='saga', random_state=42, n_jobs=-1, **params)
    elif mtype == 'XGBoost':
        from xgboost import XGBClassifier
        params = best_params_store.get((samp_key, 'XGBoost'), {})
        model = XGBClassifier(scale_pos_weight=spw, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1, verbose=0, **params)
    elif mtype == 'LightGBM':
        import lightgbm as lgb
        params = best_params_store.get((samp_key, 'LightGBM'), {})
        model = lgb.LGBMClassifier(class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1, **params)
    elif mtype == 'RandomForest':
        params = best_params_store.get((samp_key, 'RandomForest'), {})
        model = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1, **params)
    elif mtype == 'CatBoost':
        from catboost import CatBoostClassifier
        params = best_params_store.get((samp_key, 'CatBoost'), {})
        model = CatBoostClassifier(random_seed=42, verbose=0, **params)
    else:
        from xgboost import XGBClassifier
        model = XGBClassifier(scale_pos_weight=spw, use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)

    model.fit(X_tr_s, y_tr_s)
    evaluate_model(best_config['FullName'], model, X_test, y_test)
    print('\nFull dataset run complete!')
else:
    print('Skipping full dataset run (run Part 1 + Part 2 first).')


---
## 5. Phan tich Du lieu Kham pha (EDA) - Tren 1 mau dai dien

In [ ]:
df_eda = take_sample(df_full, SAMPLE_SIZE, RANDOM_STATE_SAMPLE)
print(f'EDA sample: {len(df_eda):,} records, fatality rate={df_eda["Fatality"].mean()*100:.3f}%')

### 5.1. Thong ke don bien (Univariate Analysis)

In [ ]:
num_cols = ['C_YEAR', 'C_MNTH', 'C_WDAY', 'C_HOUR', 'C_SEV', 'C_VEHS', 'P_SEX', 'P_AGE', 'V_YEAR']
df_eda[num_cols].describe().T

In [ ]:
# ---- Collisions by Year ----
fig, ax = plt.subplots()
df_eda['C_YEAR'].dropna().astype(int).value_counts().sort_index().plot(ax=ax, kind='bar', color=C0)
ax.set_xlabel('Year'); ax.set_ylabel('Count')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'collisions_by_year.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: collisions_by_year.png')

# ---- Collisions by Month ----
fig, ax = plt.subplots()
df_eda['C_MNTH'].dropna().astype(int).value_counts().sort_index().plot(ax=ax, kind='bar', color=C0)
ax.set_xlabel('Month'); ax.set_ylabel('Count')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'collisions_by_month.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: collisions_by_month.png')

# ---- Collisions by Hour ----
fig, ax = plt.subplots()
df_eda['C_HOUR'].dropna().astype(int).value_counts().sort_index().plot(ax=ax, kind='bar', color=C0)
ax.set_xlabel('Hour'); ax.set_ylabel('Count')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'collisions_by_hour.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: collisions_by_hour.png')

### 5.2. Thong ke hai bien (Bivariate Analysis)

In [ ]:
# ---- Fatality Rate by Hour ----
fig, ax = plt.subplots()
df_eda.groupby('C_HOUR')['Fatality'].mean().mul(100).plot(ax=ax, marker='o', color=C3)
ax.set_xlabel('Hour'); ax.set_ylabel('Fatality Rate (%)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_hour.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_hour.png')

# ---- Fatality Rate by Month ----
fig, ax = plt.subplots()
df_eda.groupby('C_MNTH')['Fatality'].mean().mul(100).plot(ax=ax, marker='o', color=C3)
ax.set_xlabel('Month'); ax.set_ylabel('Fatality Rate (%)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_month.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_month.png')

# ---- Fatality Rate by Day of Week ----
fig, ax = plt.subplots()
df_eda.groupby('C_WDAY')['Fatality'].mean().mul(100).plot(ax=ax, marker='o', color=C3)
ax.set_xlabel('Day of Week'); ax.set_ylabel('Fatality Rate (%)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_wday.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_wday.png')

In [ ]:
# ---- Fatality Rate by Road User Type ----
fig, ax = plt.subplots()
df_eda.groupby('P_USER')['Fatality'].mean().mul(100).plot(ax=ax, kind='bar', color=C3)
ax.set_xlabel('User Type'); ax.set_ylabel('Fatality Rate (%)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_user.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_user.png')

# ---- Top 15 Collision Configs (Highest Fatality Rate) ----
fig, ax = plt.subplots(figsize=(10, 6))
df_eda.groupby('C_CONF', observed=False)['Fatality'].mean().sort_values(ascending=False).head(15).mul(100).plot(ax=ax, kind='bar', color=C3)
ax.set_xlabel('Collision Config'); ax.set_ylabel('Fatality Rate (%)'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_conf_top15.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_conf_top15.png')

### 5.3. Phan tich da bien (Multivariate Analysis)

In [ ]:
corr_cols = ['C_YEAR', 'C_MNTH', 'C_WDAY', 'C_HOUR', 'C_SEV', 'C_VEHS', 'P_AGE', 'Fatality']
corr_matrix = df_eda[corr_cols].dropna().corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, linewidths=0.5)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, '03_correlation_matrix.png'), bbox_inches='tight'); plt.show()
print('Saved: 03_correlation_matrix.png')

---
### 5.4. Mo rong phan tich don bien (Extended Univariate)


In [ ]:
print('='*50)
print('P_USER - Road User Type Distribution')
print('='*50)
vals = df_eda['P_USER'].value_counts().sort_index()
print(f'Number of types: {len(vals)}')
print(vals.to_string())
print()

fig, ax = plt.subplots()
vals.plot(ax=ax, kind='bar', color=C0)
ax.set_xlabel('Road User Type (P_USER)'); ax.set_ylabel('Count')
for i, v in enumerate(vals.values):
    ax.text(i, v + vals.max()*0.02, f'{v:,}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'puser_distribution.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: puser_distribution.png')

In [ ]:
print('='*50)
print('C_CONF - Collision Configuration Distribution (Top 20)')
print('='*50)
vals = df_eda['C_CONF'].value_counts().sort_values(ascending=False)
print(f'Number of configurations: {len(vals)}')
print(vals.head(20).to_string())
print()

fig, ax = plt.subplots(figsize=(10, 6))
vals.head(20).plot(ax=ax, kind='bar', color=C0)
ax.set_xlabel('Collision Configuration (C_CONF)'); ax.set_ylabel('Count'); ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(vals.head(20).values):
    ax.text(i, v + vals.max()*0.02, f'{v:,}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'cconf_distribution.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: cconf_distribution.png')

In [ ]:
print('='*50)
print('V_TYPE - Vehicle Type Distribution')
print('='*50)
vals = df_eda['V_TYPE'].value_counts().sort_index()
print(f'Number of types: {len(vals)}')
print(vals.to_string())
print()

fig, ax = plt.subplots()
vals.plot(ax=ax, kind='bar', color=C0)
ax.set_xlabel('Vehicle Type (V_TYPE)'); ax.set_ylabel('Count')
for i, v in enumerate(vals.values):
    ax.text(i, v + vals.max()*0.02, f'{v:,}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'vtype_distribution.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: vtype_distribution.png')

In [ ]:
print('='*50)
print('P_SAFE - Safety Equipment Distribution')
print('='*50)
vals = df_eda['P_SAFE'].value_counts().sort_index()
print(f'Number of categories: {len(vals)}')
print(vals.to_string())
print()

fig, ax = plt.subplots()
vals.plot(ax=ax, kind='bar', color=C0)
ax.set_xlabel('Safety Equipment (P_SAFE)'); ax.set_ylabel('Count')
for i, v in enumerate(vals.values):
    ax.text(i, v + vals.max()*0.02, f'{v:,}', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'psafe_distribution.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: psafe_distribution.png')

In [ ]:
feat_cat_list = [
    ('C_WTHR', 'Weather Condition'),
    ('C_RSUR', 'Road Surface'),
    ('C_TRAF', 'Traffic Control'),
    ('C_RCFG', 'Road Configuration'),
    ('C_RALN', 'Road Alignment'),
]

for col, label in feat_cat_list:
    print(f'===== {col} - {label} =====')
    vals = df_eda[col].value_counts().sort_index()
    print(f'Number of categories: {len(vals)}')
    print(vals.to_string())
    print()

    fig, ax = plt.subplots(figsize=(8, 4))
    vals.plot(ax=ax, kind='bar', color=C0)
    ax.set_xlabel(label); ax.set_ylabel('Count')
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, f'{col.lower()}_distribution.png'), bbox_inches='tight', dpi=DPI); plt.show()
    print(f'Saved: {col.lower()}_distribution.png')
    print()

In [ ]:
print('='*50)
print('P_AGE Distribution by Fatality Status')
print('='*50)
df_eda_plot = df_eda.dropna(subset=['P_AGE']).copy()
df_eda_plot['P_AGE'] = df_eda_plot['P_AGE'].astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
for label, color, name in [(0, C0, 'Non-Fatal'), (1, C3, 'Fatal')]:
    subset = df_eda_plot[df_eda_plot['Fatality'] == label]['P_AGE']
    subset.plot(ax=ax, kind='hist', bins=80, alpha=0.6, label=f'{name} (n={len(subset):,})', color=color)
ax.set_xlabel('Age'); ax.set_ylabel('Frequency'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'page_by_fatality.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: page_by_fatality.png')

print(f'\nFatal: mean={df_eda_plot[df_eda_plot["Fatality"]==1]["P_AGE"].mean():.1f}, median={df_eda_plot[df_eda_plot["Fatality"]==1]["P_AGE"].median():.0f}')
print(f'Non-Fatal: mean={df_eda_plot[df_eda_plot["Fatality"]==0]["P_AGE"].mean():.1f}, median={df_eda_plot[df_eda_plot["Fatality"]==0]["P_AGE"].median():.0f}')

In [ ]:
print('='*50)
print('Heatmap: Collision Count by Hour x Month')
print('='*50)
hm = df_eda.pivot_table(index='C_HOUR', columns='C_MNTH', values='C_CASE', aggfunc='count')
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(hm, ax=ax, cmap='YlOrRd', annot=False, fmt='.0f', cbar_kws={'label': 'Count'})
ax.set_xlabel('Month'); ax.set_ylabel('Hour')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'heatmap_hour_month_count.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: heatmap_hour_month_count.png')

In [ ]:
print('='*50)
print('Heatmap: Collision Count by Hour x Day of Week')
print('='*50)
hm = df_eda.pivot_table(index='C_HOUR', columns='C_WDAY', values='C_CASE', aggfunc='count')
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(hm, ax=ax, cmap='YlOrRd', annot=False, fmt='.0f', cbar_kws={'label': 'Count'})
ax.set_xlabel('Day of Week'); ax.set_ylabel('Hour')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'heatmap_hour_wday_count.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: heatmap_hour_wday_count.png')

---
### 5.5. Mo rong phan tich hai bien (Extended Bivariate)


In [ ]:
print('='*50)
print('Fatality Rate by Year')
print('='*50)
fig, ax = plt.subplots()
yearly_rate = df_eda.groupby('C_YEAR')['Fatality'].mean().mul(100)
yearly_rate.plot(ax=ax, marker='o', color=C3, linewidth=2)
ax.set_xlabel('Year'); ax.set_ylabel('Fatality Rate (%)')
ax.set_ylim(0, yearly_rate.max()*1.2)
for year, rate in yearly_rate.items():
    ax.annotate(f'{rate:.2f}%', (year, rate), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=7)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_year.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_year.png')

In [ ]:
print('='*50)
print('Fatality Rate by Vehicle Type')
print('='*50)
fig, ax = plt.subplots(figsize=(9, 5))
rate = df_eda.groupby('V_TYPE')['Fatality'].mean().mul(100).sort_values(ascending=False)
rate.plot(ax=ax, kind='bar', color=C3)
ax.set_xlabel('Vehicle Type (V_TYPE)'); ax.set_ylabel('Fatality Rate (%)')
for i, v in enumerate(rate.values):
    ax.text(i, v + rate.max()*0.03, f'{v:.2f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_vtype.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_vtype.png')

In [ ]:
print('='*50)
print('Fatality Rate by Safety Equipment')
print('='*50)
fig, ax = plt.subplots()
rate = df_eda.groupby('P_SAFE')['Fatality'].mean().mul(100).sort_values(ascending=False)
rate.plot(ax=ax, kind='bar', color=C3)
ax.set_xlabel('Safety Equipment (P_SAFE)'); ax.set_ylabel('Fatality Rate (%)')
for i, v in enumerate(rate.values):
    ax.text(i, v + rate.max()*0.03, f'{v:.2f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_psafe.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_psafe.png')

In [ ]:
print('='*50)
print('Fatality Rate by Weather Condition')
print('='*50)
fig, ax = plt.subplots()
rate = df_eda.groupby('C_WTHR')['Fatality'].mean().mul(100).sort_values(ascending=False)
rate.plot(ax=ax, kind='bar', color=C3)
ax.set_xlabel('Weather Condition (C_WTHR)'); ax.set_ylabel('Fatality Rate (%)')
for i, v in enumerate(rate.values):
    ax.text(i, v + rate.max()*0.03, f'{v:.2f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_cwthr.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_cwthr.png')

In [ ]:
print('='*50)
print('Fatality Rate by Collision Severity & Age Group')
print('='*50)
df_eda_plot = df_eda.dropna(subset=['P_AGE', 'C_SEV']).copy()
df_eda_plot['P_AGE'] = df_eda_plot['P_AGE'].astype(int)
df_eda_plot['AgeGroup'] = pd.cut(df_eda_plot['P_AGE'], bins=[0, 18, 30, 45, 60, 120], labels=['0-18', '19-30', '31-45', '46-60', '60+'])
fig, ax = plt.subplots()
rate = df_eda_plot.groupby(['C_SEV', 'AgeGroup'])['Fatality'].mean().mul(100).unstack()
rate.plot(ax=ax, kind='bar', color=[C0, C2, C1, C4, C3])
ax.set_xlabel('Collision Severity (C_SEV)'); ax.set_ylabel('Fatality Rate (%)'); ax.legend(title='Age Group')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fatality_by_sev_age.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: fatality_by_sev_age.png')

---
### 5.6. Mo rong phan tich da bien (Extended Multivariate)


In [ ]:
print('='*50)
print('User Type vs Fatality (Normalized Stacked Bar)')
print('='*50)
ct = pd.crosstab(df_eda['P_USER'], df_eda['Fatality'], normalize='index')
fig, ax = plt.subplots(figsize=(9, 5))
ct.plot(ax=ax, kind='bar', stacked=True, color=[C0, C3])
ax.set_xlabel('Road User Type (P_USER)'); ax.set_ylabel('Proportion'); ax.legend(['Non-Fatal', 'Fatal'])
ax.axhline(y=df_eda['Fatality'].mean(), color='gray', ls='--', lw=1, label=f'Global rate: {df_eda["Fatality"].mean()*100:.2f}%')
for i, (idx, row) in enumerate(ct.iterrows()):
    ax.text(i, 1.02, f'{row[1]*100:.1f}%', ha='center', fontsize=8, color=C3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'puser_vs_fatality.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: puser_vs_fatality.png')

In [ ]:
print('='*50)
print('Collision Configuration vs Fatality Rate (Top 20)')
print('='*50)
rate = df_eda.groupby('C_CONF', observed=False)['Fatality'].mean().mul(100).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
rate.plot(ax=ax, kind='bar', color=C3)
ax.set_xlabel('Collision Configuration (C_CONF)'); ax.set_ylabel('Fatality Rate (%)'); ax.tick_params(axis='x', rotation=45)
for i, v in enumerate(rate.values):
    ax.text(i, v + rate.max()*0.03, f'{v:.1f}%', ha='center', fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'cconf_vs_fatality.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: cconf_vs_fatality.png')

In [ ]:
print('='*50)
print('Age Distribution by User Type (Boxplot)')
print('='*50)
df_eda_plot = df_eda.dropna(subset=['P_AGE', 'P_USER']).copy()
df_eda_plot['P_AGE'] = df_eda_plot['P_AGE'].astype(int)
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_eda_plot, x='P_USER', y='P_AGE', palette='muted', ax=ax)
ax.set_xlabel('Road User Type (P_USER)'); ax.set_ylabel('Age')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'page_by_puser_boxplot.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: page_by_puser_boxplot.png')

In [ ]:
print('='*50)
print('Fatality Rate Heatmap: Month x Hour')
print('='*50)
hm = df_eda.pivot_table(index='C_HOUR', columns='C_MNTH', values='Fatality', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(hm, ax=ax, cmap='RdYlGn_r', annot=False, fmt='.3f', cbar_kws={'label': 'Fatality Rate'})
ax.set_xlabel('Month'); ax.set_ylabel('Hour')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'heatmap_hour_month_fatality.png'), bbox_inches='tight', dpi=DPI); plt.show()
print('Saved: heatmap_hour_month_fatality.png')

---
## [Optional] Apriori Association Rules - Tren full dataset (chay rieng biet)

In [ ]:
# =============================================================================
# KHAI PHA LUAT KET HOP (APRIORI) - chay rieng tren full dataset
# =============================================================================
# from mlxtend.preprocessing import TransactionEncoder
# from mlxtend.frequent_patterns import apriori, association_rules
# 
# trans_cols = ['Fatality', 'C_CONF', 'C_WTHR', 'C_RSUR', 'C_RCFG', 'C_TRAF',
#               'C_VEHS', 'P_USER', 'P_SAFE', 'P_PSN', 'V_TYPE']
# 
# df_ap = df_full.copy()
# df_ap = story_imputation(df_ap)
# df_ap = filter_data(df_ap)
# df_ap = mice_imputation(df_ap, num_cols)
# 
# trans_clean = df_ap[trans_cols].dropna()
# transactions = [
#     [f'{col}_{val}' for col, val in zip(trans_cols, row)]
#     for row in trans_clean.astype(str).values
# ]
# te = TransactionEncoder()
# onehot = te.fit(transactions).transform(transactions, sparse=True)
# print(f'One-hot matrix: {onehot.shape}')
# 
# frequent_itemsets = apriori(onehot, min_support=0.01, use_colnames=True, max_len=4, low_memory=True)
# print(f'Frequent itemsets: {len(frequent_itemsets)}')
# 
# if len(frequent_itemsets) > 0:
#     rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.05)
#     print(f'Association rules: {len(rules)}')
#     fatal_rules = rules[rules['consequents'].apply(lambda x: 'Fatality_1' in x)]
#     fatal_rules = fatal_rules.sort_values('lift', ascending=False).head(15)
#     if len(fatal_rules) > 0:
#         print('Top 15 rules -> Fatality=1:')
#         for _, row in fatal_rules.iterrows():
#             ants = ', '.join(sorted(row['antecedents']))
#             cons = ', '.join(sorted(row['consequents']))
#             print(f'  {{{ants}}} -> {{{cons}}} (sup={row["support"]:.4f}, conf={row["confidence"]:.3f}, lift={row["lift"]:.2f})')
#     if len(rules) > 0:
#         plt.figure(figsize=(10, 6))
#         plt.scatter(rules['support'], rules['confidence'], c=rules['lift'], cmap='viridis', alpha=0.6, s=rules['lift']*10)
#         plt.colorbar(label='Lift'); plt.xlabel('Support'); plt.ylabel('Confidence')
#         plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, '04_apriori_rules.png'), bbox_inches='tight'); plt.show()
#         print('Saved: 04_apriori_rules.png')
# else:
#     print('No frequent itemsets found.')